In [1]:
# RAPORT JAKOŚCI - WCZYTANIE I PODGLĄD DANYCH

import pandas as pd
from google.colab import files

df = pd.read_csv("CENY.csv", sep=";", dtype=str)

display(df.head())

print("Wymiary danych:", df.shape)
print("Kolumny:", df.columns.tolist())

,Produkt,2019,2020,2021,2022,2023,2024,2025
0,ryż - za 1kg,"3,96","4,24","4,59","5,72","6,48","6,23","6,32"
1,bułka pszenna - za 50g,"0,48","0,53","0,56","0,70","0,79",-,-
2,bułka pszenna mała wagi do 60g – za 1 szt.,-,-,-,-,-,"0,84","0,88"
3,mąka pszenna (1) - za 1kg,-,-,-,-,"4,08","3,87","3,76"
4,chleb żytni razowy zwykły - za 1kg,"6,90",-,-,-,-,-,-


Wymiary danych: (64, 8)
Kolumny: ['Produkt', '2019', '2020', '2021', '2022', '2023', '2024', '2025']


In [2]:
# RAPORT JAKOŚCI CEN - BRAKI I DUPLIKATY

import pandas as pd

lata = [str(rok) for rok in range(2019, 2026)]

# Kopia danych surowych do kontroli
df_kontrola = df.copy()

# Zamiana oznaczeń braków na NaN
df_kontrola[lata] = df_kontrola[lata].replace(
    ["-", "", " "], pd.NA
)

print("=== PODSTAWOWE INFORMACJE ===")
print("Liczba wierszy:", len(df_kontrola))
print("Liczba kolumn:", len(df_kontrola.columns))

print("\n=== BRAKUJĄCE WARTOŚCI ===")
braki = pd.DataFrame({
    "Liczba braków": df_kontrola.isna().sum(),
    "Procent braków (%)": (
        df_kontrola.isna().mean() * 100
    ).round(2)
})
display(braki)

print("\n=== DUPLIKATY ===")
print("Identyczne wiersze:", df_kontrola.duplicated().sum())
print(
    "Powtarzające się nazwy produktów:",
    df_kontrola["Produkt"].duplicated().sum()
)

print("\n=== PUSTE NAZWY PRODUKTÓW ===")
print(df_kontrola["Produkt"].isna().sum())

=== PODSTAWOWE INFORMACJE ===
Liczba wierszy: 64
Liczba kolumn: 8

=== BRAKUJĄCE WARTOŚCI ===


,Liczba braków,Procent braków (%)
Produkt,1,1.56
2019,17,26.56
2020,17,26.56
2021,17,26.56
2022,17,26.56
2023,16,25.00
2024,18,28.12
2025,18,28.12



=== DUPLIKATY ===
Identyczne wiersze: 0
Powtarzające się nazwy produktów: 0

=== PUSTE NAZWY PRODUKTÓW ===
1


In [3]:
# RAPORT JAKOŚCI CEN - SZCZEGÓŁOWA KONTROLA BRAKÓW

print("=== WIERSZE BEZ NAZWY PRODUKTU ===")
display(df_kontrola[df_kontrola["Produkt"].isna()])

print("\n=== LICZBA BRAKÓW DLA KAŻDEGO PRODUKTU ===")

braki_produkty = df_kontrola.copy()
braki_produkty["Liczba braków"] = braki_produkty[lata].isna().sum(axis=1)

display(
    braki_produkty[
        ["Produkt", "Liczba braków"]
    ].sort_values("Liczba braków", ascending=False)
)

print("\n=== PRODUKTY BEZ ŻADNEJ CENY ===")

display(
    df_kontrola[
        df_kontrola[lata].isna().all(axis=1)
    ]
)

=== WIERSZE BEZ NAZWY PRODUKTU ===


,Produkt,2019,2020,2021,2022,2023,2024,2025
63,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



=== LICZBA BRAKÓW DLA KAŻDEGO PRODUKTU ===


,Produkt,Liczba braków
63,NaN,7
4,chleb żytni razowy zwykły - za 1kg,6
50,polędwica lub szynka drobiowa - za 1kg,6
49,"owoce mrożone - za 0,5kg",6
51,kiełbasa - za 1 kg,6
...,...,...
43,czekolada mleczna - za 100g,0
48,"mieszanka mrożona marchew i groszek - za 0,5kg",0
59,"herbata czarna, liściasta (1) - za 100g",0
57,sok jabłkowy - za 1l,0



=== PRODUKTY BEZ ŻADNEJ CENY ===


,Produkt,2019,2020,2021,2022,2023,2024,2025
63,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# RAPORT JAKOŚCI CEN - POPRAWNOŚĆ CEN

lata = [str(rok) for rok in range(2019, 2026)]

ceny = df[lata].copy()

# Zamiana przecinków na kropki i braków na NaN
for kolumna in lata:
    ceny[kolumna] = pd.to_numeric(
        ceny[kolumna].astype(str)
        .str.strip()
        .str.replace(",", ".", regex=False),
        errors="coerce"
    )

print("=== NIEPRAWIDŁOWE WARTOŚCI CEN ===")

wyniki = pd.DataFrame({
    "Rok": lata,
    "Braki": ceny.isna().sum().values,
    "Ceny zerowe": (ceny == 0).sum().values,
    "Ceny ujemne": (ceny < 0).sum().values,
    "Najniższa cena": ceny.min().values,
    "Najwyższa cena": ceny.max().values
})

display(wyniki)

print("\n=== WARTOŚCI NIEBĘDĄCE LICZBAMI ===")

for rok in lata:
    nieprawidlowe = (
        df[rok].notna()
        & ~df[rok].astype(str).str.strip().isin(["", "-"])
        & ceny[rok].isna()
    )

    if nieprawidlowe.any():
        print(f"\nRok {rok}:")
        display(df.loc[nieprawidlowe, ["Produkt", rok]])

=== NIEPRAWIDŁOWE WARTOŚCI CEN ===


,Rok,Braki,Ceny zerowe,Ceny ujemne,Najniższa cena,Najwyższa cena
0,2019,17,0,0,0.48,33.35
1,2020,17,0,0,0.53,37.01
2,2021,17,0,0,0.56,37.85
3,2022,17,0,0,0.70,42.43
4,2023,16,0,0,0.79,47.94
5,2024,18,0,0,0.84,50.31
6,2025,18,0,0,0.88,52.02



=== WARTOŚCI NIEBĘDĄCE LICZBAMI ===


In [5]:
# RAPORT JAKOŚCI CEN - KOMPLETNOŚĆ DANYCH

kompletne = ceny.notna().all(axis=1)
liczba_lat = ceny.notna().sum(axis=1)

print("=== KOMPLETNOŚĆ PRODUKTÓW ===")
print("Produkty z cenami we wszystkich 7 latach:", kompletne.sum())
print("Produkty z brakującymi cenami:", (~kompletne).sum())

print("\n=== LICZBA DOSTĘPNYCH LAT DLA PRODUKTÓW ===")
display(liczba_lat.value_counts().sort_index())

print("\n=== PRODUKTY Z KOMPLETEM CEN ===")
display(df.loc[kompletne, ["Produkt"] + lata])

=== KOMPLETNOŚĆ PRODUKTÓW ===
Produkty z cenami we wszystkich 7 latach: 31
Produkty z brakującymi cenami: 33

=== LICZBA DOSTĘPNYCH LAT DLA PRODUKTÓW ===


,count
0,1
1,4
2,4
3,9
4,7
5,4
6,4
7,31



=== PRODUKTY Z KOMPLETEM CEN ===


,Produkt,2019,2020,2021,2022,2023,2024,2025
0,ryż - za 1kg,"3,96","4,24","4,59","5,72","6,48","6,23","6,32"
6,"chleb pszenno-żytni - za 0,5kg","2,66","2,91","3,17","4,16","4,82","5,13","5,38"
8,makaron jajeczny - za 400g,"4,26","4,48","4,67","5,49","6,42","6,60","6,49"
10,"kasza jęczmienna - za 0,5kg","2,38","2,53","2,68","3,31","3,80","3,85","3,84"
11,"kasza gryczana, prażona - za 0,5kg","4,59","4,83","5,44","7,59","8,49","7,35","6,86"
14,mięso wieprzowe bez kości (schab) - za 1 kg,"18,99","19,17","17,99","21,84","25,73","26,02","25,04"
15,mięso wieprzowe bez kości (łopatka) - za 1kg,"14,99","16,12","14,65","17,38","20,77","20,25","19,44"
16,kurczęta patroszone - za 1kg,"7,46","7,35","8,55","11,69","11,76","11,93","13,24"
19,kiełbasa sucha - za 1kg,"33,35","37,01","37,85","42,43","47,94","50,31","52,02"
21,boczek wędzony - za 1kg,"24,79","27,91","27,74","30,79","36,03","38,04","39,45"


In [ ]:
# RAPORT JAKOŚCI - NAJDŁUŻSZE PRZERWY

pb95 = df_czas[
    df_czas["productName"] == "Pb95"
].sort_values("effectiveDate").copy()

pb95["Poprzednia data"] = pb95["effectiveDate"].shift(1)

pb95["Przerwa (dni)"] = (
    pb95["effectiveDate"] - pb95["Poprzednia data"]
).dt.days

najdluzsze = pb95[
    pb95["Przerwa (dni)"] >= 5
][["Poprzednia data", "effectiveDate", "Przerwa (dni)"]]

display(najdluzsze)

,Poprzednia data,effectiveDate,Przerwa (dni)
1864,2019-04-19,2019-04-24,5.0
1857,2019-05-03,2019-05-08,5.0
1625,2020-04-10,2020-04-15,5.0
1449,2020-12-24,2020-12-30,6.0
1383,2021-04-02,2021-04-07,5.0
1343,2021-06-03,2021-06-08,5.0
1201,2021-12-24,2021-12-30,6.0
1126,2022-04-15,2022-04-20,5.0
1117,2022-04-30,2022-05-05,5.0
1096,2022-06-02,2022-06-07,5.0


In [ ]:
# RAPORT JAKOŚCI - SPRAWDZENIE JEDNOSTEK

print(df["unit"].value_counts(dropna=False))

unit
PLN/m3    5829
Name: count, dtype: int64


In [ ]:
# RAPORT JAKOŚCI - DATY Z PRZYSZŁOŚCI

print("Daty z przyszłości:")
print((daty > pd.Timestamp.today()).sum())

Daty z przyszłości:
0


In [ ]:
# RAPORT JAKOŚCI - SPRAWDZENIE CENY ZA LITR

roznica = (df["cena_za_litr"] - df["value"] / 1000).abs()

print("Nieprawidłowe przeliczenia:", (roznica > 0.000001).sum())
print("Największa różnica:", roznica.max())


Nieprawidłowe przeliczenia: 0
Największa różnica: 0.0
